# Week 4 · Notebook 2 — Backtesting & Financial Metrics
### Multi-Agent Forecasting Project

**Name:**  
**Date:**  

---

A model that looks great on one 2019 test set means nothing. To trust a strategy we must simulate trading it **forward through history**, retraining as new data arrives — a **walk-forward backtest** — and judge it with the metrics traders actually care about: **Sharpe ratio**, **maximum drawdown**, **directional accuracy**.

This notebook assembles the agents (Notebooks 2–3) and the Hedge referee (Notebook 1) into one honest evaluation.

**Topics covered:**
1. Financial metrics: **Sharpe**, **max drawdown**, **directional accuracy**, **information ratio**
2. Turning a prediction into a **trading position** and an **equity curve**
3. The **walk-forward** (expanding window) backtest loop
4. Running agents + Hedge across **2013–2024**
5. Reading the results table honestly

> To keep this notebook fast in Colab we use the three quick agents (Trend, Momentum, Volatility). The LSTM plugs in identically — just slower.

In [ ]:
# Run this cell first.
!pip -q install yfinance xgboost statsmodels python-dateutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('Setup complete.')

In [ ]:
# Rebuild the feature matrix, agents, and Hedge from earlier notebooks (condensed).
import yfinance as yf
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from xgboost import XGBRegressor

def _rsi(s, p=14):
    d=s.diff(); g=d.clip(lower=0); l=-d.clip(upper=0)
    ag=g.ewm(com=p-1,min_periods=p).mean(); al=l.ewm(com=p-1,min_periods=p).mean()
    return 100-100/(1+ag/al.replace(0,np.nan))
def _macd(s):
    line=s.ewm(span=12,adjust=False).mean()-s.ewm(span=26,adjust=False).mean()
    return line.ewm(span=9,adjust=False).mean()
def _boll(s,w=20):
    m=s.rolling(w).mean(); sd=s.rolling(w).std(); return ((m+2*sd)-(m-2*sd))/m
def build_feature_matrix(df):
    c=df['Close'].shift(1)
    cols={'rsi_14':_rsi(c),'macd_signal':_macd(c),'bb_width':_boll(c)}
    for l in (1,2,3,5,10): cols[f'lag_{l}']=df['log_return'].shift(l)
    for w in (5,21):
        cols[f'rolling_mean_{w}']=df['log_return'].shift(1).rolling(w).mean()
        cols[f'rolling_std_{w}']=df['log_return'].shift(1).rolling(w).std()
    cols['vix_level']=df['vix_close'].shift(1); cols['vix_change_5d']=df['vix_change_5d'].shift(1)
    cols['log_return']=df['log_return']
    return pd.DataFrame(cols).dropna()

spy=yf.download('SPY',start='2010-01-01',end='2024-12-31',auto_adjust=True,progress=False)[['Close']]
spy.columns=['Close']; spy['log_return']=np.log(spy['Close']/spy['Close'].shift(1))
vix=yf.download('^VIX',start='2010-01-01',end='2024-12-31',auto_adjust=True,progress=False)[['Close']]
vix.columns=['vix_close']; vix['vix_change_5d']=vix['vix_close'].pct_change(5)
feature_df=build_feature_matrix(spy.join(vix,how='inner').dropna())

MOM=['lag_1','lag_2','lag_3','lag_5','lag_10','rolling_mean_5','rolling_mean_21','rolling_std_5','rsi_14','macd_signal']
VOL=['bb_width','rolling_std_5','rolling_std_21','vix_level','vix_change_5d']
class TrendAgent:
    name='TrendAgent'
    def __init__(self): self._m=LinearRegression(); self._dp=None
    def fit(self,tr):
        self._dp=DeterministicProcess(index=tr.index,constant=True,order=1,additional_terms=[CalendarFourier(freq='YE',order=4)],drop=True)
        self._m.fit(self._dp.in_sample(),tr['log_return'].values)
    def predict(self,te): return self._m.predict(self._dp.out_of_sample(steps=len(te),forecast_index=te.index))
class _XGBAgent:
    def __init__(self,cols): self.cols=cols; self._m=XGBRegressor(n_estimators=200,learning_rate=0.05,max_depth=4,subsample=0.8,colsample_bytree=0.8,random_state=42,verbosity=0)
    def fit(self,tr): self._m.fit(tr[self.cols],tr['log_return'].values)
    def predict(self,te): return self._m.predict(te[self.cols])
class MomentumAgent(_XGBAgent):
    name='MomentumAgent'
    def __init__(self): super().__init__(MOM)
class VolatilityAgent(_XGBAgent):
    name='VolatilityAgent'
    def __init__(self): super().__init__(VOL)
class HedgeAggregator:
    def __init__(self,n,eta=0.2,alpha=0.05,loss_mode='mse'):
        self.n=n; self.eta=eta; self.alpha=alpha; self.loss_mode=loss_mode
        self.weights=np.ones(n)/n; self.weight_history=[]
    def aggregate(self,p): return float(np.dot(self.weights,p))
    def update(self,p,a):
        p=np.array(p,float)
        if self.loss_mode=='directional':
            ls=-(np.sign(p)*a); lo,hi=ls.min(),ls.max(); nm=(ls-lo)/(hi-lo+1e-12)
        else:
            ls=(p-a)**2; nm=ls/(ls.mean()+1e-12)
        self.weights*=np.exp(-self.eta*nm); self.weights/=self.weights.sum()
        self.weights=(1-self.alpha)*self.weights+self.alpha/self.n; self.weights/=self.weights.sum()
        self.weight_history.append(self.weights.copy())
    def reset(self): self.weights=np.ones(self.n)/self.n; self.weight_history.clear()
print('Loaded feature_df', feature_df.shape, 'and all agent/aggregator classes.')

---
## Section 1 — Financial Metrics

Prediction error (MAE) is not what a trader cares about. They care about **risk-adjusted return**. The four core metrics:

| Metric | Question it answers | Formula sketch |
|---|---|---|
| **Sharpe ratio** | Return per unit of risk? | `mean(returns)/std(returns) * sqrt(252)` |
| **Max drawdown** | Worst peak-to-trough loss? | `max((peak - value)/peak)` |
| **Directional accuracy** | How often is the sign right? | `mean(sign(pred) == sign(actual))` |
| **Information ratio** | Excess return vs a benchmark per unit of tracking error | `mean(active)/std(active) * sqrt(252)` |

The `* sqrt(252)` *annualises* a daily metric (252 trading days per year).

### Exercise 1.1 — Implement the Metrics

Fill in the four functions following the formula sketches above.

In [ ]:
# YOUR CODE HERE
def sharpe_ratio(returns, annualization=252):
    returns = np.asarray(returns, float)
    if returns.std() == 0: return 0.0
    return ...                       # mean/std * sqrt(annualization)

def max_drawdown(equity_curve):
    curve = np.asarray(equity_curve, float)
    peak = np.maximum.accumulate(curve)
    drawdown = ...                   # (peak - curve) / peak
    return float(drawdown.max())

def directional_accuracy(y_true, y_pred):
    return ...                       # mean of sign(y_true) == sign(y_pred)

def information_ratio(strategy, benchmark, annualization=252):
    active = np.asarray(strategy) - np.asarray(benchmark)
    if active.std() == 0: return 0.0
    return float(active.mean() / active.std() * np.sqrt(annualization))

print('Metrics defined.')

### Exercise 1.2 — From Prediction to Equity Curve

A forecast becomes a **trade**: go long (+1) if you predict up, short (−1) if you predict down. Your daily strategy return is `position * actual_return`. Compounding those gives an **equity curve** (start with $10,000). Complete `metrics_table`.

In [ ]:
# YOUR CODE HERE
def metrics_table(y_true, y_pred, benchmark=None, label='Model'):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    position = ...                                   # np.sign(y_pred): +1 long / -1 short
    strategy_returns = position * y_true
    equity_curve = 10_000 * np.exp(np.cumsum(strategy_returns))
    row = {
        'Label': label,
        'MAE': round(float(np.mean(np.abs(y_true - y_pred))), 6),
        'Directional Accuracy': round(directional_accuracy(y_true, y_pred), 4),
        'Sharpe Ratio': round(sharpe_ratio(strategy_returns), 3),
        'Max Drawdown': round(max_drawdown(equity_curve), 4),
    }
    if benchmark is not None:
        row['Information Ratio'] = round(information_ratio(strategy_returns, benchmark), 3)
    return pd.DataFrame([row])

print('metrics_table defined.')

---
## Section 2 — The Walk-Forward Backtest

The engine: start with 3 years of training data. Fit all agents, predict the next 3 months, and step the Hedge weights forward day by day using the realised return. Then **expand** the training window by 3 months and repeat — all the way to 2024. At no point does the model see the future.

Study the provided loop, then fill the two blanks: the per-day ensemble prediction and the Hedge update.

### Exercise 2.1 — Complete the Backtest Loop

In [ ]:
# YOUR CODE HERE (fill the two marked blanks)
def walk_forward_backtest(agents, aggregator, feature_df, initial_train_years=3, step_months=3):
    aggregator.reset()
    names = [a.name for a in agents]
    start = feature_df.index[0]
    train_end = start + relativedelta(years=initial_train_years) - relativedelta(days=1)
    records, fold = [], 0

    while True:
        test_start = train_end + relativedelta(days=1)
        test_end = min(train_end + relativedelta(months=step_months), feature_df.index[-1])
        if test_start > feature_df.index[-1]: break
        train_df = feature_df.loc[:train_end]
        test_df = feature_df.loc[test_start:test_end]
        if len(train_df) < 50 or len(test_df) == 0: break

        for a in agents: a.fit(train_df)
        preds_all = {n: a.predict(test_df) for n, a in zip(names, agents)}

        for i, (date, r) in enumerate(test_df.iterrows()):
            actual = r['log_return']
            step_preds = [float(preds_all[n][i]) for n in names]
            ensemble = ...                       # aggregator.aggregate(step_preds)
            ...                                  # aggregator.update(step_preds, actual)
            rec = {'date': date, 'actual': actual, 'ensemble_pred': ensemble, 'fold': fold}
            for n, p in zip(names, step_preds): rec[f'{n}_pred'] = p
            for n, w in zip(names, aggregator.weights): rec[f'{n}_weight'] = w
            records.append(rec)

        train_end += relativedelta(months=step_months); fold += 1
    return pd.DataFrame(records).set_index('date')

print('Backtest engine defined.')

### Exercise 2.2 — Run It

Build the three agents and a `HedgeAggregator`, then run the backtest over the full `feature_df`. This may take a minute or two.

In [ ]:
# YOUR CODE HERE
agents = [TrendAgent(), MomentumAgent(), VolatilityAgent()]
aggregator = HedgeAggregator(n_agents=len(agents), eta=0.2, alpha=0.05, loss_mode='mse')

results = walk_forward_backtest(agents, aggregator, feature_df)
print('Backtest rows:', len(results))
results.head()

---
## Section 3 — The Results Table

Now compare every agent, the Hedge ensemble, an equal-weight ensemble, and a buy-and-hold baseline on the financial metrics. Buy-and-hold = always long (always predict the market goes up).

### Exercise 3.1 — Build the Comparison Table

For each agent column (`<name>_pred`), the ensemble, the equal-weight mean of agent predictions, and buy-and-hold, call `metrics_table` and concatenate. Sort by Sharpe ratio, descending.

In [ ]:
# YOUR CODE HERE
bh = results['actual'].values        # benchmark = buy & hold daily returns
tables = []
names = [a.name for a in agents]
for n in names:
    tables.append(metrics_table(results['actual'].values, results[f'{n}_pred'].values, benchmark=bh, label=n))
tables.append(metrics_table(results['actual'].values, results['ensemble_pred'].values, benchmark=bh, label='Hedge Ensemble'))
eq = results[[f'{n}_pred' for n in names]].mean(axis=1).values
tables.append(metrics_table(results['actual'].values, eq, benchmark=bh, label='Equal Weight'))
tables.append(metrics_table(results['actual'].values, np.abs(results['actual'].values), benchmark=bh, label='Buy & Hold'))

report = pd.concat(tables, ignore_index=True).sort_values('Sharpe Ratio', ascending=False).reset_index(drop=True)
report

### Exercise 3.2 — Equity Curves

Plot the compounding equity curve ($10,000 start) of the **Hedge ensemble** vs **buy-and-hold** over the full backtest. Who ends higher, and who has the gentler drawdowns?

In [ ]:
# YOUR CODE HERE
ens_pos = np.sign(results['ensemble_pred'].values)
ens_curve = 10_000 * np.exp(np.cumsum(ens_pos * results['actual'].values))
bh_curve  = 10_000 * np.exp(np.cumsum(results['actual'].values))

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(results.index, ens_curve, label='Hedge Ensemble')
ax.plot(results.index, bh_curve, label='Buy & Hold', alpha=0.8)
ax.set_title('Equity curve: $10,000 invested'); ax.set_ylabel('Portfolio value ($)')
ax.legend(); plt.tight_layout(); plt.show()

### Exercise 3.3 — Read It Honestly

1. Did the Hedge ensemble beat buy-and-hold on **Sharpe**? On **max drawdown**? It is completely fine (and common) if buy-and-hold wins on raw return — explain *why* a lower-drawdown strategy can still be valuable.
2. Directional accuracy for all models is probably near **0.5**. Why is beating 0.5 by even a little still meaningful over thousands of trades?
3. The backtest expands the training window and never reuses future data. Point to the exact line that guarantees the test window comes *after* the training window.

**Your answers:**

1. 

2. 

3. 

In [ ]:
# Save results for the final notebook (the dashboard reads these).
results.to_csv('backtest_results.csv')
report.to_csv('metrics_report.csv', index=False)
print('Saved backtest_results.csv and metrics_report.csv')

---
## Bonus Challenge ⭐

- **A.** Add the LSTM `SequenceAgent` from Notebook 3 to the `agents` list and re-run. Does the extra diversity move the ensemble Sharpe? (Expect a longer runtime.)
- **B.** Re-run with `loss_mode='directional'` in the Hedge. Do the agent weights end up more concentrated or more even than with `'mse'`?
- **C.** Plot the Hedge **weight history** (`results[[f'{n}_weight' for n in names]]`) as a stacked area chart. Can you spot the weights shifting around the 2020 crash?

In [ ]:
# BONUS — YOUR CODE HERE


---
## Submission Checklist

- [ ] All four metric functions and `metrics_table` are implemented
- [ ] The walk-forward backtest runs end-to-end and produces a results DataFrame
- [ ] The comparison table is sorted by Sharpe and includes Hedge, Equal-Weight, and Buy & Hold
- [ ] The equity-curve plot renders
- [ ] `backtest_results.csv` and `metrics_report.csv` are saved
- [ ] All short-answer cells are filled in
- [ ] The notebook runs top-to-bottom with **Runtime → Restart and run all**
- [ ] Your name and date are filled in at the top

**Submit as:** `Week4_2_Backtesting_Metrics_YourName.ipynb`

> **Final notebook:** assemble everything into a shareable product — an interactive dashboard, a clean repo, and a mini prediction competition.